  # Step 3: Data Preprocessing & Feature Engineering


### 3.1  Feature Engineering
Create Amount_log = np.log1p(Amount) — the Amount distribution is heavily right-skewed.

Create Hour = (Time % 86400) // 3600 — extract the hour of day from the Time column (seconds since first transaction).

Drop the original Time and Amount columns after creating the above features.

In [1]:
import numpy as np



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_excel("creditcard.xlsx", engine="openpyxl")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0     0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1     0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2     1 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3     1 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4     2 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

    

In [4]:
df.columns = df.columns.str.strip()

print(df.columns)

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


In [5]:
sample_df, _ = train_test_split(
    df,
    train_size=50000,
    stratify=df["Class"],
    random_state=42
)

print(sample_df.shape)

(50000, 31)


In [6]:
sample_df["Amount_log"] = np.log1p(sample_df["Amount"])

print(sample_df[["Amount", "Amount_log"]].head())

        Amount  Amount_log
29784    38.31    3.671479
196949   37.89    3.660737
137061   70.00    4.262680
53715   145.50    4.987025
42739     1.00    0.693147


In [7]:
sample_df["Hour"] = ((sample_df["Time"] % 86400) // 3600).astype(int)

print(sample_df[["Time", "Hour"]].head())

          Time  Hour
29784    35628     9
196949  131771    12
137061   81998    22
53715    46114    12
42739    41224    11


In [8]:
sample_df = sample_df.drop(columns=["Time", "Amount"])

print(sample_df.head())
print(sample_df.columns)

              V1        V2        V3        V4        V5        V6        V7  \
29784  -1.370413  1.095589 -0.668018 -0.024219  1.675652  3.823629 -0.634559   
196949  1.805238  0.961264 -1.717212  4.094625  0.938666 -0.227785  0.152911   
137061 -1.789123  0.941366  1.432897 -1.318458 -0.450124 -1.290731  0.933642   
53715  -0.711379 -1.431688  0.630874 -2.551985  0.509280 -0.565154  0.258866   
42739   1.303287  1.023966 -3.187599  0.468391  3.353988  2.431416  0.184510   

              V8        V9       V10  ...       V22       V23       V24  \
29784   1.836666 -0.485368 -0.547920  ... -0.609795  0.027077  0.986022   
196949  0.066753 -1.073784  0.334537  ... -0.450959  0.098530 -0.662272   
137061 -0.306360  0.917978  0.722699  ... -0.202022 -0.017159  0.740137   
53715  -0.289951 -2.680067  1.154872  ... -1.064836  0.435373 -0.932473   
42739   0.622721 -0.556254 -1.485591  ... -0.823366 -0.241908  0.674554   

             V25       V26       V27       V28  Class  Amount_log  H

### 3.2  Scaling

Apply StandardScaler to Amount_log and Hour only — V1–V28 are already PCA-scaled.



In [9]:
from sklearn.preprocessing import StandardScaler

In [10]:

scaler = StandardScaler()

sample_df[["Amount_log", "Hour"]] = scaler.fit_transform(
    sample_df[["Amount_log", "Hour"]]
)

print(sample_df[["Amount_log", "Hour"]].head())

        Amount_log      Hour
29784     0.313613 -0.866491
196949    0.307105 -0.354251
137061    0.671760  1.353215
53715     1.110565 -0.354251
42739    -1.490648 -0.524998


In [11]:
print(sample_df[["Amount_log", "Hour"]].describe())

         Amount_log          Hour
count  5.000000e+04  5.000000e+04
mean  -3.808509e-16  1.401190e-16
std    1.000010e+00  1.000010e+00
min   -1.910553e+00 -2.403211e+00
25%   -7.448517e-01 -6.957446e-01
50%   -1.108444e-02  1.579884e-01
75%    7.230648e-01  8.409748e-01
max    3.560589e+00  1.523961e+00



### 3.3  Train-Test Split

Split: 80% train / 20% test with stratify=y, random_state=42.

Print class distribution in both train and test sets. Confirm stratification preserved the fraud ratio.



In [12]:

X = sample_df.drop("Class", axis=1)

y = sample_df["Class"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (50000, 30)
y Shape: (50000,)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

Training Set Shape: (40000, 30)
Testing Set Shape: (10000, 30)


In [14]:
print("Original Dataset Distribution")
print(y.value_counts(normalize=True))

print("\nTraining Dataset Distribution")
print(y_train.value_counts(normalize=True))

print("\nTesting Dataset Distribution")
print(y_test.value_counts(normalize=True))

Original Dataset Distribution
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64

Training Dataset Distribution
Class
0    0.998275
1    0.001725
Name: proportion, dtype: float64

Testing Dataset Distribution
Class
0    0.9983
1    0.0017
Name: proportion, dtype: float64


In [15]:
print("Training Class Counts")
print(y_train.value_counts())

print("\nTesting Class Counts")
print(y_test.value_counts())

Training Class Counts
Class
0    39931
1       69
Name: count, dtype: int64

Testing Class Counts
Class
0    9983
1      17
Name: count, dtype: int64


In [16]:
print("Original Fraud Ratio :", round(y.mean() * 100, 4), "%")
print("Train Fraud Ratio    :", round(y_train.mean() * 100, 4), "%")
print("Test Fraud Ratio     :", round(y_test.mean() * 100, 4), "%")

if abs(y.mean() - y_train.mean()) < 0.0001 and abs(y.mean() - y_test.mean()) < 0.0001:
    print("\n Stratification preserved successfully.")
else:
    print("\n Stratification was not preserved.")

Original Fraud Ratio : 0.172 %
Train Fraud Ratio    : 0.1725 %
Test Fraud Ratio     : 0.17 %

 Stratification preserved successfully.


### 3.4  Imbalance Handling — Three Strategies

Prepare three versions of the training set:

Original (imbalanced): X_train, y_train — no modification.

SMOTE: Apply SMOTE(random_state=42, sampling_strategy=0.1) to X_train — bring fraud up to 10% of majority class. Name: X_smote, y_smote.

Undersampled: Use RandomUnderSampler(random_state=42, sampling_strategy=0.1) on X_train. Name: X_under, y_under.

Print class counts for all three versions.



In [17]:

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [18]:

X_original = X_train.copy()
y_original = y_train.copy()

print("Original Training Set")
print(y_original.value_counts())

Original Training Set
Class
0    39931
1       69
Name: count, dtype: int64


In [19]:

smote = SMOTE(
    random_state=42,
    sampling_strategy=0.1
)

X_smote, y_smote = smote.fit_resample(
    X_train,
    y_train
)

print("SMOTE Training Set")
print(y_smote.value_counts())

SMOTE Training Set
Class
0    39931
1     3993
Name: count, dtype: int64


In [20]:

under = RandomUnderSampler(
    random_state=42,
    sampling_strategy=0.1
)

X_under, y_under = under.fit_resample(
    X_train,
    y_train
)

print("Under Sampled Training Set")
print(y_under.value_counts())

Under Sampled Training Set
Class
0    690
1     69
Name: count, dtype: int64


In [21]:
print("="*40)
print("Original Training Set")
print(y_original.value_counts())

print("="*40)
print("SMOTE Training Set")
print(y_smote.value_counts())

print("="*40)
print("Under Sampled Training Set")
print(y_under.value_counts())

Original Training Set
Class
0    39931
1       69
Name: count, dtype: int64
SMOTE Training Set
Class
0    39931
1     3993
Name: count, dtype: int64
Under Sampled Training Set
Class
0    690
1     69
Name: count, dtype: int64


In [22]:
import pandas as pd

comparison = pd.DataFrame({
    "Original": y_original.value_counts(),
    "SMOTE": y_smote.value_counts(),
    "UnderSample": y_under.value_counts()
})

comparison.index = ["Legitimate (0)", "Fraud (1)"]

print(comparison)

                Original  SMOTE  UnderSample
Legitimate (0)     39931  39931          690
Fraud (1)             69   3993           69
